# Dynamic vs. Static Token Merging for Visual Document Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohail0992/Static-vs-Dynamic-Token-Merging-for-Visual-Document-Retrieval/blob/main/Static_vs_Dynamic_Token_Merging_for_Visual_Document_Retrieval.ipynb)
*One click to run: the badge opens this notebook in Google Colab; the setup cell fetches the evaluation documents from GitHub automatically. Use a GPU runtime (Runtime -> Change runtime type -> T4).*

**Question.** Light-ColPali (Ma et al., ACL Findings 2025) shrinks ColPali's index by merging each page's
~1 030 patch embeddings with one **fixed** merging factor. The paper observes that text-dense pages lose more accuracy
than sparse ones and leaves *adaptive* factors as future work. Here we test a **content-aware (dynamic) factor**
against the paper's static baseline.

| Method | What it does | Source |
|---|---|---|
| **Full** | No merging (reference, 100 %) | ColPali v1.2 |
| **Static 9** | Same factor on every page: the paper's Light-ColPali operating point | official `colpali-engine` `HierarchicalTokenPooler` |
| **Thesis Dynamic** | Factor chosen per page from its visual density `S` (2 = dense … 49 = sparse) | ours |
| **Static (memory-matched)** | Static factor giving the *same total index size* as Dynamic: the fair control | official `HierarchicalTokenPooler` |

**Experiments**

* **A. ViDoRe corpus (54 real PDF documents, 68 pages):** single- and multi-page documents from 5 datasets from the paper's own benchmark: text-rich *DocVQA*, *TAT-DQA* and
  low-density *InfoVQA*, *ArxivQA*, *TabFQuAD*. Every page has a real query, so we measure true retrieval accuracy (**NDCG@5**).
* **B. The paper itself (13 pages):** pages of `2025.findings-acl.1003.pdf` with 8 hand-labelled queries (case study).

**Read the results carefully**

* The paper's numbers (97.8 % of ColPali's NDCG@5 at factor 9, ~11 % memory) come from models **fine-tuned with merging**.
  We merge the off-the-shelf ColPali v1.2 (*training-free*), so absolute retention will be lower. Compare methods
  **with each other**, not against the paper's absolute figures.
* Dynamic and Static 9 use *different amounts of memory*. Always read accuracy together with memory,
  and use the memory-matched control to judge whether the dynamic factor adds value.

**Sections:** 1 Setup · 2 Configuration & model · 3 Core functions · 4 Benchmark A · 5 Benchmark B · 6 Final comparison

## Background: the concepts this notebook relies on

**How ColPali retrieves documents.** ColPali is a vision-language model that never runs OCR. It looks at a page as an
image, cuts it into a grid of small patches, and turns **each patch** into a 128-dimensional vector ("patch embedding").
One page therefore becomes ~**1 030 vectors**, not one. A text query is embedded the same way, one vector per query token.

**MaxSim scoring (late interaction).** To score how well a page answers a query, each query-token vector finds its
single best-matching patch vector on the page (the *max* similarity), and these maxima are summed. A page is relevant
if *every part of the question* finds *some patch* that matches it. This is why ColPali is accurate - and why it is
expensive: all ~1 030 vectors per page must be kept in the index.

**Token merging (what Light-ColPali does).** Patches on a page are redundant - background, margins, repeated textures.
Merging groups similar patch vectors into clusters (hierarchical/Ward clustering on their similarity) and keeps one
averaged vector per cluster. The **merging factor F** controls how aggressive this is: factor 9 keeps roughly
1 030 / 9 ≈ 114 vectors per page, i.e. ~11 % of the memory. Bigger F = smaller index = more information lost.

**The thesis idea (dynamic F).** A fixed F treats a dense 2-column text page and a near-empty title slide identically.
Our hypothesis: estimate how *visually dense* each page is (score `S` from pixel statistics), then merge dense pages
gently (small F, keep detail) and sparse pages aggressively (large F, save memory). The paper explicitly names this
adaptive choice of F as an open problem ("we leave this as future work", Section 6).

**How we measure success.**
* **NDCG@5** - for each query, where does its correct page rank among all pages in the corpus? Rank 1 scores 1.0,
  rank 2 ≈ 0.63, below rank 5 scores 0. We report the mean over all queries, and "retained %" relative to the
  unmerged model (the paper's convention).
* **Recall@1 / MRR** - stricter views of the same ranking.
* **Memory** - total stored vectors x 128 dims x 2 bytes (fp16), reported as % of the unmerged index.

A method "wins" only if it gets **more accuracy for the same memory** (or equal accuracy for less) - never from one
number alone.

## 1. Setup
Installs the official ColPali repo (`colpali-engine`) plus the PDF/analysis dependencies (see `requirements.txt` / `README.md` in this folder). Run once; if pip upgrades `transformers`, restart the runtime and run again from the top. On Colab, also upload the `evaluation_docs/` folder and the ACL PDF next to this notebook.

In [ ]:
import os
if not os.path.exists("evaluation_docs"):          # fresh Colab runtime: fetch notebook + data from GitHub
    !git clone -q https://github.com/sohail0992/Static-vs-Dynamic-Token-Merging-for-Visual-Document-Retrieval.git _repo
    %cd _repo

!pip install -q colpali-engine pdf2image scipy pandas matplotlib tqdm
!apt-get install -y -qq poppler-utils > /dev/null 2>&1 || true   # pdf renderer (Colab/Linux; on macOS: brew install poppler)

## 2. Configuration & model
All experiment settings live in this one cell, so changing the static factor, the number of pages or the datasets never requires touching the logic below.

* `MODEL_NAME` - the official **ColPali v1.2** checkpoint, loaded through `colpali_engine` (the paper authors' repo).
* `pooler` - the repo's own `HierarchicalTokenPooler`: Ward clustering on patch-similarity, the exact merging operation behind the Light-ColPali numbers. Using it (instead of re-implementing clustering) is what makes our Static 9 row a faithful reproduction of the paper's baseline.
* `STATIC_FACTOR = 9` - the paper's headline operating point (Table 2: ~11 % memory, 97.8 % retained NDCG@5 after fine-tuning).
* `EVAL_DIR` - the prepared 50-page evaluation corpus (see Section 4 and `README.md`).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageStat
from tqdm.auto import tqdm
from pdf2image import convert_from_path
from colpali_engine.models import ColPali, ColPaliProcessor
from colpali_engine.compression.token_pooling import HierarchicalTokenPooler

# ---- Experiment settings ---------------------------------------------------
MODEL_NAME = "vidore/colpali-v1.2"
STATIC_FACTOR = 9          # paper baseline: Light-ColPali operating point (11.4 % relative memory in Table 2)
K = 5                      # NDCG@K, same metric as the paper
EMBED_DIM = 128            # ColPali embedding size
BYTES_PER_VALUE = 2        # embeddings stored as 16-bit floats (paper, footnote 1)

EVAL_DIR = Path("evaluation_docs")     # 50 real PDF documents + manifest.csv (file, dataset, page_type, query)
RESULTS_DIR = Path("results")          # everything the run produces (tables, plots) goes here, never into evaluation_docs/
RESULTS_DIR.mkdir(exist_ok=True)
PDF_CANDIDATES = ["2025.findings-acl.1003.pdf", "/content/2025.findings-acl.1003.pdf", "hafsa/2025.findings-acl.1003.pdf"]

# One colour per method role (validated categorical palette: blue/orange/aqua + neutral grey).
ROLE_COLORS = {"full": "#8a8a85", "static": "#2a78d6", "dynamic": "#eb6834", "matched": "#1baf7a"}

# ---- Model -----------------------------------------------------------------
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
print(f"Loading {MODEL_NAME} on {device} ({dtype})...")
model = ColPali.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map=device).eval()
processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
pooler = HierarchicalTokenPooler()     # official token merging from illuin-tech/colpali

## 3. Core functions
Everything both benchmarks share. Reading top to bottom:

* **Thesis logic (ours).** `get_density_score(image)` returns `S` in [0, 1]: the grayscale pixel standard deviation of the page, normalised by 128. Busy pages (lots of ink, figures, colour) score high; near-blank pages score low. `get_thesis_factor(S)` maps that linearly to a merging factor: `F = 49 - 47*S`, clipped to [2, 49] - i.e. densest page -> F=2 (gentle merging), emptiest page -> F=49 (aggressive merging). These two functions ARE the thesis contribution; everything else is shared infrastructure.
* **Embedding.** `embed_images` / `embed_queries` run the official model once per page/query and cache the raw (n_tokens x 128) embeddings on CPU, so all four methods merge and score the *same* embeddings - no method gets a different model run.
* **Merging.** `pool_page(embedding, factor)` is a thin wrapper around the official pooler; factor 1 means 'keep everything'.
* **Scoring.** `maxsim_matrix` computes the late-interaction score of every query against every page; `retrieval_metrics` turns the score matrix into NDCG@5, Recall@1 and MRR by ranking the correct page among all pages.
* **`run_benchmark`.** The one driver both experiments call. It evaluates Full, Static 9 and Thesis Dynamic, then computes which single static factor would use the same total memory as Dynamic and evaluates that too (the *memory-matched control*). Returns the summary table, per-dataset NDCG, and a per-page table of S / assigned factor / token counts.
* **`plot_benchmark`.** Two panels: NDCG@5 per dataset (bars, one colour per method), and the accuracy-vs-memory trade-off (each method one dot; up and to the left is better).

In [ ]:
# ---- Thesis logic: visual density -> dynamic merging factor ----------------
def get_density_score(image):
    # Visual density S in [0, 1]: grayscale pixel standard deviation / 128.
    return min(1.0, ImageStat.Stat(image.convert("L")).stddev[0] / 128.0)


def get_thesis_factor(S):
    # Linear mapping: S = 1 (dense) -> factor 2, S = 0 (sparse) -> factor 49.
    return int(max(2, min(49, round(49 - 47 * S))))


# ---- Embedding (official ColPali) -------------------------------------------
@torch.no_grad()
def embed_images(images, batch_size=4):
    # One (n_tokens, 128) float32 CPU tensor per page.
    out = []
    for i in tqdm(range(0, len(images), batch_size), desc="Embedding pages"):
        batch = processor.process_images(images[i:i + batch_size]).to(model.device)
        out.extend(model(**batch).float().cpu().unbind(0))
    return out


@torch.no_grad()
def embed_queries(queries, batch_size=16):
    # One (n_query_tokens, 128) float32 CPU tensor per query.
    out = []
    for i in range(0, len(queries), batch_size):
        batch = processor.process_queries(queries[i:i + batch_size]).to(model.device)
        out.extend(model(**batch).float().cpu().unbind(0))
    return out


# ---- Token merging (official hierarchical pooling) --------------------------
def pool_page(embedding, factor):
    # Merge one page's patch embeddings into ~n/factor vectors (factor 1 = unchanged).
    if factor <= 1:
        return embedding
    return pooler.pool_embeddings([embedding], pool_factor=int(factor))[0]


# ---- Retrieval scoring -------------------------------------------------------
def maxsim_matrix(q_embs, d_embs):
    # Late-interaction score s(q, p) = sum over query tokens of max over page tokens (paper, Section 3).
    scores = torch.zeros(len(q_embs), len(d_embs))
    for j, d in enumerate(d_embs):
        for i, q in enumerate(q_embs):
            scores[i, j] = (q @ d.T).max(dim=1).values.sum()
    return scores


def retrieval_metrics(scores, relevant, k=K):
    # Per-query NDCG@k, Recall@1 and MRR for single-relevant-page queries.
    rows = torch.arange(len(relevant))
    rel_scores = scores[rows, torch.as_tensor(relevant)]
    rank = (1 + (scores > rel_scores[:, None]).sum(dim=1)).numpy()      # rank 1 = best
    return {
        f"ndcg@{k}": np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0),
        "recall@1": (rank == 1).astype(float),
        "mrr": 1.0 / rank,
    }


# ---- One benchmark = one corpus scored under every method -------------------
def run_benchmark(d_embs, q_embs, relevant, groups, S):
    # Scores Full / Static / Dynamic / memory-matched Static on one corpus.
    # d_embs: page embeddings; q_embs: query embeddings; relevant[i]: page index of query i;
    # groups[i]: dataset label of query i; S: density score per page.
    n_full = np.array([len(e) for e in d_embs])
    dyn_factors = np.array([get_thesis_factor(s) for s in S])
    groups = np.asarray(groups)

    def evaluate(factors):
        pooled = [pool_page(e, f) for e, f in tqdm(list(zip(d_embs, factors)), desc="Merging", leave=False)]
        return np.array([len(p) for p in pooled]), retrieval_metrics(maxsim_matrix(q_embs, pooled), relevant)

    methods = [
        ("Full (no merging)", "full", np.ones(len(d_embs), dtype=int)),
        (f"Static {STATIC_FACTOR} (paper)", "static", np.full(len(d_embs), STATIC_FACTOR)),
        ("Thesis Dynamic", "dynamic", dyn_factors),
    ]
    results = {}
    for name, role, factors in methods:
        tokens, metrics = evaluate(factors)
        results[name] = dict(role=role, factors=factors, tokens=tokens, metrics=metrics)

    # Fair control: a static factor that uses the same total memory as Dynamic.
    matched = max(1, round(n_full.sum() / results["Thesis Dynamic"]["tokens"].sum()))
    if matched != STATIC_FACTOR:
        name = f"Static {matched} (matched to Dynamic)"
        factors = np.full(len(d_embs), matched)
        tokens, metrics = evaluate(factors)
        results[name] = dict(role="matched", factors=factors, tokens=tokens, metrics=metrics)

    ndcg_key = f"ndcg@{K}"
    full_ndcg = results["Full (no merging)"]["metrics"][ndcg_key].mean()
    summary, by_group = [], {}
    for name, r in results.items():
        m = r["metrics"]
        summary.append({
            "Method": name,
            "Effective factor": n_full.sum() / r["tokens"].sum(),
            "Tokens / page": r["tokens"].mean(),
            "Memory vs full (%)": 100 * r["tokens"].sum() / n_full.sum(),
            "Index size (KB)": r["tokens"].sum() * EMBED_DIM * BYTES_PER_VALUE / 1024,
            f"NDCG@{K}": m[ndcg_key].mean(),
            f"NDCG@{K} retained (%)": 100 * m[ndcg_key].mean() / full_ndcg,
            "Recall@1": m["recall@1"].mean(),
            "MRR": m["mrr"].mean(),
        })
        by_group[name] = pd.Series(m[ndcg_key]).groupby(groups).mean()

    pages = pd.DataFrame({"group": groups_per_page(groups, relevant, len(d_embs)), "S": S, "dynamic factor": dyn_factors,
                          "full tokens": n_full, "dynamic tokens": results["Thesis Dynamic"]["tokens"]})
    return {"summary": pd.DataFrame(summary).set_index("Method"),
            "by_group": pd.DataFrame(by_group),
            "pages": pages,
            "roles": {n: r["role"] for n, r in results.items()}}


def groups_per_page(groups, relevant, n_pages):
    # Dataset label of each page (taken from the query that targets it).
    labels = [""] * n_pages
    for g, p in zip(groups, relevant):
        labels[p] = g
    return labels


# ---- Plot: accuracy per dataset + the accuracy/memory trade-off ---------------
def plot_benchmark(res, title):
    by_group, summary, roles = res["by_group"], res["summary"], res["roles"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2.2, 1]})

    width = 0.8 / len(by_group.columns)
    x = np.arange(len(by_group.index))
    for i, name in enumerate(by_group.columns):
        ax1.bar(x + i * width - 0.4 + width / 2, by_group[name], width * 0.9, label=name, color=ROLE_COLORS[roles[name]])
    ax1.set_xticks(x)
    ax1.set_xticklabels(by_group.index)
    ax1.set_ylabel(f"NDCG@{K}")
    ax1.set_ylim(0, 1.05)
    ax1.set_title("Retrieval accuracy by dataset")

    # Dynamic and its memory-matched control share (almost) the same x by construction: draw the larger marker first.
    marker_size = {"dynamic": 260, "static": 120, "full": 120, "matched": 70}
    for name, row in summary.iterrows():
        ax2.scatter(row["Memory vs full (%)"], row[f"NDCG@{K} retained (%)"], s=marker_size[roles[name]],
                    color=ROLE_COLORS[roles[name]], edgecolor="white", linewidth=2, zorder=3 if roles[name] != "dynamic" else 2)
    ax2.set_xlabel("Index memory vs full (%)")
    ax2.set_ylabel(f"NDCG@{K} retained vs full (%)")
    ax2.set_title("Accuracy vs. memory trade-off")

    for ax in (ax1, ax2):
        ax.grid(True, axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False, fontsize=10)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(rect=(0, 0.08, 1, 0.96))
    fname = RESULTS_DIR / (title.split(":")[0].strip().lower().replace(" ", "_") + ".png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()

## 4. Benchmark A: ViDoRe corpus (50 real pages)
The statistical experiment. `evaluation_docs/` ships with this notebook and holds **54 real PDF documents** (50 single-page + 4 multi-page of 4-5 pages, 68 pages in total) drawn from five ViDoRe test splits: text-rich *DocVQA* and *TAT-DQA*, low-density *InfoVQA*, *ArxivQA* and *TabFQuAD*. `manifest.csv` is the ground truth: one row per (document, page) with the **real benchmark query whose answer is on that page**. Exactly as in production VDR, the pipeline renders each PDF to page images itself and feeds those to ColPali - the page, not the file, is the retrieval unit. Open the PDFs to see exactly which documents the numbers refer to.

All 50 pages are pooled into ONE retrieval corpus: when we ask query #17, the system must rank page #17 above the other 49 pages, which include both similar pages from the same dataset and very different ones. A merging method that destroys discriminative detail will visibly push correct pages down this ranking. (The old notebook scored one generic query against each page separately - that measures nothing, because there was nothing to retrieve *against*.)

In [ ]:
# Load the evaluation corpus: render every PDF document to page images (ColPali's indexing unit is the page).
# manifest.csv: one row per (document, page) with the real query whose answer is on that page.
manifest = pd.read_csv(EVAL_DIR / "manifest.csv")
rendered = {f: convert_from_path(EVAL_DIR / f, dpi=150)
            for f in tqdm(manifest["file"].unique(), desc="Rendering PDFs")}
pages_a = [
    {"doc": r.file, "dataset": r.dataset, "page_type": r.page_type, "query": r.query,
     "image": rendered[r.file][r.page].convert("RGB")}
    for r in manifest.itertuples()
]
n_docs = manifest["file"].nunique()
print(f"{n_docs} documents -> {len(pages_a)} pages loaded from {EVAL_DIR}/ "
      f"({(manifest.groupby('file').size() > 1).sum()} documents are multi-page)")
display(manifest.groupby(["dataset", "page_type"]).agg(documents=("file", "nunique"), pages=("file", "size")).reset_index())

In [ ]:
# Embed pages + queries once; every method below reuses these embeddings.
images_a = [p["image"] for p in pages_a]
d_embs_a = embed_images(images_a)
q_embs_a = embed_queries([p["query"] for p in pages_a])
S_a = np.array([get_density_score(img) for img in images_a])

res_a = run_benchmark(d_embs_a, q_embs_a, relevant=np.arange(len(pages_a)),
                      groups=[p["dataset"] for p in pages_a], S=S_a)

In [ ]:
# Results: summary table, per-dataset accuracy, density/factor assignment, plots.
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Overall (54 documents, 68-page corpus)")
display(res_a["summary"])
print(f"\nNDCG@{K} per dataset")
display(res_a["by_group"])
print("\nDensity score S and the factor Thesis Dynamic assigned, per dataset")
display(res_a["pages"].groupby("group", sort=False).agg(S=("S", "mean"), dynamic_factor=("dynamic factor", "mean")))
plot_benchmark(res_a, "Benchmark A: 54 ViDoRe documents (68 pages)")

## 5. Benchmark B: the paper itself (case study)
The 13 pages of `2025.findings-acl.1003.pdf` form a corpus. Each query below is answered by one specific page (0-based index, checked against the PDF). With only 8 queries this is a qualitative case study, not a statistical result.

In [ ]:
pdf_path = next((p for p in PDF_CANDIDATES if os.path.exists(p)), None)
if pdf_path is None:
    raise FileNotFoundError(f"Upload 2025.findings-acl.1003.pdf next to the notebook (looked in {PDF_CANDIDATES}).")

pdf_pages = convert_from_path(pdf_path, dpi=150)
print(f"{pdf_path}: {len(pdf_pages)} pages")

# (query, 0-based index of the page that answers it)
PDF_QUERIES = [
    ("How does memory usage compare with NDCG@5 for ColPali, Light-ColPali and DSE?",           0),   # Figure 1
    ("Why does random token pruning outperform score-oriented and attention-oriented pruning?",   3),   # Section 4.2-4.3, Fig. 2-3
    ("Retrieval performance versus merging factor for 1D pooling, 2D pooling and clustering",     4),   # Figure 4
    ("Retrieval performance of pre-encoder, post-encoder, post-LLM and post-projector merging",   6),   # Table 1
    ("NDCG@5 of Light-ColPali and Light-ColQwen2 at merging factors 4, 9, 25 and 49",             7),   # Table 2
    ("Training and embedding generation time of ColQwen2 versus Light-ColQwen2 on A100 GPUs",     7),   # Table 3
    ("Prompt used to synthesize five questions from a document page",                             11),  # Listing 1
    ("NDCG@5 of retrievers built on Qwen2-VL-7B",                                                 12),  # Table 4
]

In [ ]:
d_embs_b = embed_images(pdf_pages)
q_embs_b = embed_queries([q for q, _ in PDF_QUERIES])
S_b = np.array([get_density_score(img) for img in pdf_pages])

res_b = run_benchmark(d_embs_b, q_embs_b, relevant=[p for _, p in PDF_QUERIES],
                      groups=["ACL paper"] * len(PDF_QUERIES), S=S_b)
res_b["pages"]["group"] = [f"p.{i + 1}" for i in range(len(pdf_pages))]

print("Overall (13-page corpus, 8 queries)")
display(res_b["summary"])
print("\nPer-page density and dynamic factor")
display(res_b["pages"].set_index("group")[["S", "dynamic factor", "full tokens", "dynamic tokens"]])
plot_benchmark(res_b, "Benchmark B: pages of the ACL paper")

## 6. Final comparison: Thesis Dynamic vs. the paper's static baseline
One table for both benchmarks. The three questions it answers:

1. **Dynamic vs. Static 9:** accuracy *and* memory, since the two use different budgets.
2. **Dynamic vs. memory-matched Static:** the fair test of whether adapting the factor to each page helps, at equal index size.
3. **Where does dynamic help or hurt?** The per-dataset NDCG@5 above (the paper reports that text-rich pages suffer most).

In [ ]:
cols = ["Effective factor", "Tokens / page", "Memory vs full (%)", f"NDCG@{K}", f"NDCG@{K} retained (%)", "Recall@1", "MRR"]
final = pd.concat({"A: ViDoRe (54 docs / 68 pages)": res_a["summary"][cols], "B: ACL paper (13 pages)": res_b["summary"][cols]},
                  names=["Benchmark"])
final.to_csv(RESULTS_DIR / "final_comparison.csv")
res_a["summary"].to_csv(RESULTS_DIR / "benchmark_A_summary.csv")
res_a["by_group"].to_csv(RESULTS_DIR / "benchmark_A_per_dataset.csv")
res_a["pages"].to_csv(RESULTS_DIR / "benchmark_A_pages.csv", index=False)
res_b["summary"].to_csv(RESULTS_DIR / "benchmark_B_summary.csv")
print(f"Tables saved to {RESULTS_DIR}/")
display(final)

for bench, res in {"A: ViDoRe": res_a, "B: ACL paper": res_b}.items():
    s = res["summary"]
    static, dyn = s.loc[f"Static {STATIC_FACTOR} (paper)"], s.loc["Thesis Dynamic"]
    print(f"\n{bench}")
    print(f"  Dynamic vs Static {STATIC_FACTOR}: NDCG@{K} {dyn[f'NDCG@{K}'] - static[f'NDCG@{K}']:+.3f}, "
          f"memory {dyn['Memory vs full (%)']:.1f}% vs {static['Memory vs full (%)']:.1f}% of full")
    matched = [n for n in s.index if "matched" in n]
    if matched:
        m = s.loc[matched[0]]
        print(f"  Dynamic vs {matched[0]} (same memory): NDCG@{K} {dyn[f'NDCG@{K}'] - m[f'NDCG@{K}']:+.3f}")